# TRKR Notebook

Runs a TRKR delay scan in corrected `t_cor_ps` coordinates.

Review `config_path`, then connect devices explicitly; this notebook does not auto-connect hardware.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from kohdalab.api import Experiment, format_point, load_config, make_trkr_live_update, trkr_plan


config_path: Path | None = None  # Set a lab-specific JSON path before hardware use.
config = load_config() if config_path is None else load_config(config_path)
experiment = Experiment(config, auto_connect=False)
# Review config, then run experiment.connect_all() before acquiring data.
zero = config["measurements"]["move_abs"]["zero"]

def print_status(status: str) -> None:
    print(f"status: {status}", flush=True)


In [ ]:
minimum_ps = -5.0
maximum_ps = 5.0
step_ps = 5.0
wait_s = 0.0
return_to_zero = True
output = None  # None uses the output path configured for trkr.
y_key = "R_V"

plan = trkr_plan(
    minimum_ps=minimum_ps,
    maximum_ps=maximum_ps,
    step_ps=step_ps,
    t_zero_ps=float(zero.get("t_ps", 0.0)),
    coordinate="measurement",
)
live_update = make_trkr_live_update(y_key=y_key, title="TRKR")
axis_key = "t_cor_ps"


In [ ]:
def handle_point(point):
    print(format_point(point, axis_key=axis_key), flush=True)
    live_update(point)


rows = experiment.run_trkr(
    plan=plan,
    wait_s=wait_s,
    return_to_zero=return_to_zero,
    output=output,
    on_status=print_status,
    on_point=handle_point,
)

display(pd.DataFrame(rows))


In [ ]:
# Run this when you are done with the notebook session.
# experiment.disconnect_all()
